# Example 0.13.2 / 3: Exercise 7-1: Restricting Multicommodity Data

Companion notebook for **0.13.2 Exercises on Sets, Parameters and Variables** in the AMPL lab Documentation, section "Exercise 7-1: Restricting Multicommodity Data". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exercise 7-1: Restricting Multicommodity Data

Exercise 7-1 lists conditions that the data of a multicommodity transportation model must satisfy,
and asks you to state each one as a restriction phrase or a `check`
(Restrictions, Defaults and Checks (AMPL lab documentation, section 1.2.1)). A condition on one value fits a restriction
phrase; a condition that relates several values needs `check`.

| Condition | Statement |
|---|---|
| No city is both an origin and a destination | `check: card(ORIG inter DEST) = 0;` |
| More destinations than origins | `check: card(DEST) > card(ORIG);` |
| Demand at a destination does not exceed 1000 (here: all products together) | `check {j in DEST}: sum {p in PROD} demand[j,p] <= 1000;` |
| Supply equals demand for each product | `check {p in PROD}: sum {i in ORIG} supply[i,p] = sum {j in DEST} demand[j,p];` |
| Supply equals demand over all products | `check: sum {i in ORIG, p in PROD} supply[i,p] = sum {j in DEST, p in PROD} demand[j,p];` |
| An origin's supply fits its shipping limits | `check {i in ORIG}: sum {p in PROD} supply[i,p] <= sum {j in DEST} limit[i,j];` |
| A destination's demand fits its shipping limits | `check {j in DEST}: sum {p in PROD} demand[j,p] <= sum {i in ORIG} limit[i,j];` |

If the exercise means 1000 for each product separately, the restriction phrase
`param demand {DEST, PROD} >= 0, <= 1000;` states it. The model below includes all seven checks
with a small data set of its own.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ORIG;                                # origins
set DEST;                                # destinations
set PROD;                                # products
param supply {ORIG, PROD} >= 0;          # tons available
param demand {DEST, PROD} >= 0;          # tons required
param limit {ORIG, DEST} >= 0;           # tons of all products on one link
param cost {ORIG, DEST, PROD} >= 0;      # dollars per ton

check: card(ORIG inter DEST) = 0;        # no city is both origin and destination
check: card(DEST) > card(ORIG);          # more destinations than origins
check {j in DEST}: sum {p in PROD} demand[j,p] <= 1000;  # demand per city
check {p in PROD}: sum {i in ORIG} supply[i,p] = sum {j in DEST} demand[j,p];  # balance per product
check: sum {i in ORIG, p in PROD} supply[i,p] = sum {j in DEST, p in PROD} demand[j,p];  # total balance
check {i in ORIG}: sum {p in PROD} supply[i,p] <= sum {j in DEST} limit[i,j];  # origin limits
check {j in DEST}: sum {p in PROD} demand[j,p] <= sum {i in ORIG} limit[i,j];  # destination limits

var Trans {ORIG, DEST, PROD} >= 0;       # tons shipped

minimize Total_Cost: sum {i in ORIG, j in DEST, p in PROD} cost[i,j,p] * Trans[i,j,p];  # dollars

subject to Supply {i in ORIG, p in PROD}: sum {j in DEST} Trans[i,j,p] = supply[i,p];   # ship all supply
subject to Demand {j in DEST, p in PROD}: sum {i in ORIG} Trans[i,j,p] = demand[j,p];   # meet demand
subject to Multi {i in ORIG, j in DEST}: sum {p in PROD} Trans[i,j,p] <= limit[i,j];    # link limit

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

ampl.set["ORIG"] = ["GARY", "CLEV"]
ampl.set["DEST"] = ["FRA", "DET", "LAN"]
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["supply"] = {("GARY", "bands"): 300, ("GARY", "coils"): 700,
                        ("CLEV", "bands"): 600, ("CLEV", "coils"): 1200}
ampl.param["demand"] = {("FRA", "bands"): 300, ("FRA", "coils"): 650,
                        ("DET", "bands"): 300, ("DET", "coils"): 600,
                        ("LAN", "bands"): 300, ("LAN", "coils"): 650}
ampl.param["limit"] = {(i, j): 700 for i in ["GARY", "CLEV"] for j in ["FRA", "DET", "LAN"]}
ampl.param["cost"] = {
    ("GARY", "FRA", "bands"): 30, ("GARY", "FRA", "coils"): 39,
    ("GARY", "DET", "bands"): 10, ("GARY", "DET", "coils"): 14,
    ("GARY", "LAN", "bands"): 8, ("GARY", "LAN", "coils"): 11,
    ("CLEV", "FRA", "bands"): 22, ("CLEV", "FRA", "coils"): 27,
    ("CLEV", "DET", "bands"): 7, ("CLEV", "DET", "coils"): 9,
    ("CLEV", "LAN", "bands"): 10, ("CLEV", "LAN", "coils"): 12,
}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, ampl.obj["Total_Cost"].value())

HiGHS 1.14.0: optimal solution; objective 44800
7 simplex iterations
0 barrier iterations
solved 44800.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ORIG := GARY CLEV;
set DEST := FRA DET LAN;
set PROD := bands coils;
param supply:  bands  coils :=
GARY             300    700
CLEV             600   1200;
param demand:  bands  coils :=
FRA              300    650
DET              300    600
LAN              300    650;
param limit:  FRA  DET  LAN :=
GARY          700  700  700
CLEV          700  700  700;
param cost :=
[*,*,bands]:  FRA  DET  LAN :=
GARY           30   10    8
CLEV           22    7   10
[*,*,coils]:  FRA  DET  LAN :=
GARY           39   14   11
CLEV           27    9   12;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Total_Cost;

HiGHS 1.14.0: optimal solution; objective 44800
7 simplex iterations
0 barrier iterations
Total_Cost = 44800

